# Comparisons with real stars

This tutorial compares AsteroScale posterior predictions with published stellar properties. The literature mass, radius and luminosity values are retained only for comparison and are **not** passed to the solver.

Agreement should be approximate: scaling relations are not expected to reproduce detailed modelling, dynamical masses or interferometric radii at their quoted precision.

## Sources

- **α Cen A:** seismic inputs from Kjeldsen et al. (2005); comparison properties from [Kervella et al. (2017)](https://doi.org/10.1051/0004-6361/201629505).
- **ε Tau:** seismic inputs and comparison properties from [Arentoft et al. (2019)](https://doi.org/10.1051/0004-6361/201834690).
- **ε Indi A:** seismic inputs from [Campante et al. (2024)](https://doi.org/10.1051/0004-6361/202449197); comparison mass from [Lundkvist et al. (2024)](https://doi.org/10.3847/1538-4357/ad25f2).

The first two are rough agreement checks. ε Indi A is deliberately retained as a stress case: its global scaling-relation mass is significantly below the interferometry-assisted literature value.

In [ ]:
import numpy as np
import asteroscale as ast

In [ ]:
REAL_STARS = {
    "alpha Cen A": {
        "given": {"Teff": (5795, 19), "FeH": (0.23, 0.05),
                  "numax": (2410, 130), "dnu": (106.0, 0.5)},
        "literature": {"M": (1.1055, 0.0039), "R": (1.2234, 0.0053),
                       "L": (1.521, 0.015)},
    },
    "epsilon Tau": {
        "given": {"Teff": (4950, 70), "FeH": (0.15, 0.05),
                  "numax": (56.4, 1.1), "dnu": (5.00, 0.01)},
        "literature": {"M": (2.458, 0.073), "R": (12.46, 0.26),
                       "L": (79.4, 3.4)},
    },
    "epsilon Indi A": {
        "given": {"Teff": (4649, 7), "FeH": (-0.17, 0.05),
                  "numax": (5305, 176), "dnu": (201.25, 0.16)},
        "literature": {"M": (0.782, 0.023)},
    },
}

In [ ]:
def compare_with_literature(name, samples, literature):
    print(f"\n{name}")
    print(f"{'quantity':10s} {'AsteroScale p50 [p16, p84]':34s} literature")
    print("-" * 72)
    for quantity, (reference, error) in literature.items():
        p16, p50, p84 = np.percentile(samples[quantity], [16, 50, 84])
        print(
            f"{quantity:10s} {p50:8.3f} [{p16:8.3f}, {p84:8.3f}]"
            f"       {reference:8.3f} +/- {error:.3f}"
        )

In [ ]:
literature_results = {}
for index, (name, data) in enumerate(REAL_STARS.items()):
    quantities = list(data["literature"])
    solver = ast.Solver(preset="fast", seed=100 + index)
    samples = solver.solve(data["given"], want=quantities)
    literature_results[name] = samples
    compare_with_literature(name, samples, data["literature"])

The ε Indi A discrepancy is scientifically useful: a successful and numerically precise run is not evidence that an empirical scaling relation is accurate for every star. External validation remains essential.